In [14]:
from pathlib import Path
import pandas as pd
from PIL import Image
import numpy as np
from keras import Sequential, Input, layers
from keras.applications.efficientnet_v2 import EfficientNetV2B0
from keras.applications.efficientnet_v2 import preprocess_input, decode_predictions

In [8]:
def load_train_df():
    df = pd.read_csv("../../datasets/food-101-100-images/meta/train.txt", header=None)
    return df

In [9]:
def load_test_df():
    df = pd.read_csv("../../datasets/food-101-100-images/meta/test.txt", header=None)
    return df

In [10]:
load_train_df()

,0
0,apple_pie/1005649
1,apple_pie/1014775
2,apple_pie/1026328
3,apple_pie/1028787
4,apple_pie/1043283
...,...
7586,waffles/1343456
7587,waffles/1351305
7588,waffles/1353542
7589,waffles/1354919


In [11]:
load_test_df()

,0
0,apple_pie/1011328
1,apple_pie/101251
2,apple_pie/1034399
3,apple_pie/103801
4,apple_pie/1038694
...,...
2504,waffles/1311578
2505,waffles/1313096
2506,waffles/1336006
2507,waffles/1343475


# Parse image

In [ ]:
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

def parse_image(df):
    images = []
    labels = []
    for path in df[0]:
        class_name = Path(path).parent.name
        img_path = f"../../datasets/food-101/images/{path}.jpg"
        img = Image.open(img_path).convert('RGB').resize((224,224))
        images.append(np.array(img, dtype=np.float32) / 255.0)
        labels.append(class_name)

    le = LabelEncoder()
    y = le.fit_transform(labels)

    X = np.stack(images)
    return X, y

In [13]:
train_X, train_y = parse_image(load_train_df())

In [ ]:
train_X.shape

In [ ]:
preds = model.predict(x)
# decode the results into a list of tuples (class, description, probability)
# (one such list for each sample in the batch)
print('Predicted:', decode_predictions(preds, top=3)[0])
# Predicted: [(u'n02504013', u'Indian_elephant', 0.82658225), (u'n01871265', u'tusker', 0.1122357), (u'n02504458', u'African_elephant', 0.061040461)]


In [ ]:
def efficient_net_model():
    model = EfficientNetV2B0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    model.trainable = False

    return model

In [ ]:
def initialise_model():
    model = Sequential()

    model.add(Input(shape=(224,224,3)))
    model.add(efficient_net_model())

    model.add(layers.Flatten())

    model.add(layers.Dense(100, activation='relu'))
    model.add(layers.Dense(200, activation='relu'))

    model.add(layers.Dense(101, activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    return model

In [ ]:
model = initialise_model()
model.summary()

In [ ]:
model.fit(train_X, train_y, epochs=1)

In [ ]:
preds = model.predict()